# campod coverage: what the four cameras see, by tilt

Ground and side coverage of the four campods for a range of camera tilts, from the lenses fitted on
real stills. For each tilt: how many cameras see each ground point, and how many see it at a given
GSD or better -- on the ground, and on the camera-facing side of a tree trunk at ground level and
partway up.

Lenses: each pod's Kannala-Brandt fit from ODM self-calibration (`opensfm/stats/stats.json`), averaged
over `run_dirs`. Mounts: azimuth and position from `config`; the tilt is swept. Lens and mount
conventions: `mapping_sim/lens.py`, `mapping_sim/rig.py`. Summary and the mount geometry it
answers: `docs/campod.md`, "Aim geometry".

In [ ]:
run_dirs = [
    "/home/jovyan/datasets/mapping-sim/runs/flight-2026-11-07-crash",
    "/home/jovyan/datasets/mapping-sim/runs/flight-2026-11-07-crash-dataflash",
    "/home/jovyan/datasets/mapping-sim/runs/flight-2026-11-07-crash-dataflash-30k-ultra",
]
config = "analysis/mapping_sim/configs/as-built.yaml"  # azimuths and positions; its depression is replaced by each tilt below
depressions_deg = [69.5, 65, 60, 55, 50]  # optical axis below horizontal; 69.5 is the wedge as built
height_m = 20.0
gsd_m = 0.02
trunk_heights_m = [0.0, 10.0]  # points on a trunk, metres above the ground
extent_m = 40.0

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import io, json, pathlib, sys
import yaml
from IPython.display import display as ipy_display, Image as ipy_image

sys.path.insert(0, 'analysis/mapping_sim')  # run from the repo root
from lens import KannalaBrandt
from rig import mount_rotation

plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})

def show(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    ipy_display(ipy_image(data=buf.getvalue()))

## Lenses

OpenSfM normalises by the longer image side: pixel focal = `focal * max(W, H)`, principal point =
centre + `c * max(W, H)`. One row per pod per run; the mean per pod is what the maps use.

In [ ]:
KEYS = ('focal_x', 'focal_y', 'c_x', 'c_y', 'k1', 'k2', 'k3', 'k4')
rows = []
for d in run_dirs:
    for name, c in json.loads((pathlib.Path(d) / 'cameras.json').read_text()).items():
        rows.append({'run': pathlib.Path(d).name, 'pod': name.split()[1].removeprefix('campod-'),
                     'width': c['width'], 'height': c['height'], **{k: c[k] for k in KEYS}})
fits = pd.DataFrame(rows)
W, H = int(fits.width.iloc[0]), int(fits.height.iloc[0])
S = max(W, H)
mean = fits.groupby('pod')[list(KEYS)].mean()
lenses = {pod: KannalaBrandt(W, H, m.focal_x * S, m.focal_y * S, W / 2 + m.c_x * S, H / 2 + m.c_y * S,
                             (m.k1, m.k2, m.k3, m.k4)) for pod, m in mean.iterrows()}

def edge_angles(L):
    r = lambda u, v: L.unproject(np.array([[u, v]], float))[0][0]
    deg = lambda v: np.degrees(np.arctan(v))
    l, rt, t, b = r(0, L.cy), r(W, L.cy), r(L.cx, 0), r(L.cx, H)
    corner = max(np.degrees(np.arccos(r(u, v)[2])) for u in (0, W) for v in (0, H))
    return {'focal px': L.fx, 'left deg': deg(-l[0] / l[2]), 'right deg': deg(rt[0] / rt[2]),
            'top deg': deg(-t[1] / t[2]), 'bottom deg': deg(b[1] / b[2]), 'corner off-axis deg': corner}

fits['focal px'] = fits.focal_x * S
ipy_display(fits[['run', 'pod', 'focal px', 'c_x', 'c_y', 'k1', 'k2', 'k3']].round(4))
ipy_display(pd.DataFrame({pod: edge_angles(L) for pod, L in lenses.items()}).T.round(1))

## Mounts

FLU body frame (x forward, y left, z up). Azimuth and position from `config`.

In [ ]:
cfg = yaml.safe_load(pathlib.Path(config).read_text())
mounts = {c['name']: (c['azimuth_deg'], np.array(c['position_m'], float)) for c in cfg['rig']['cameras']}
assert set(mounts) == set(lenses), (sorted(mounts), sorted(lenses))
ipy_display(pd.DataFrame({n: {'azimuth deg': a, 'x m': p[0], 'y m': p[1], 'z m': p[2]} for n, (a, p) in mounts.items()}).T)

## Method

A grid of points `height_m` below the vehicle (or on a trunk, `trunk_heights_m` above the ground).
A camera sees a point when the point projects inside its image. Its GSD there is
`range * pixel angle / cos(incidence)`: the pixel angle is `KannalaBrandt.pixel_angle` at that
off-axis angle (the larger of radial and tangential), and the incidence is against the surface's
normal -- vertical for the ground, horizontal and facing the camera for a trunk, since a round trunk
always has such a face. For the ground this is `range^2 / height * pixel angle`, so a point's GSD
depends on where it is, not on the tilt; the tilt only decides whether the point is in frame.

Not modelled: occlusion, blur, rolling shutter, exposure. Everything scales with height: at twice
the height, every distance and every GSD doubles.

In [ ]:
g = np.linspace(-extent_m, extent_m, 321)
X, Y = np.meshgrid(g, g)  # X right, Y forward
R = np.hypot(X, Y)

def seen(dep, below, surface, gsd_max=np.inf):
    pts = np.stack([Y, -X, np.full_like(X, -below)], -1)
    n = np.zeros(X.shape, int)
    for pod, L in lenses.items():
        az, t = mounts[pod]
        Rbc = mount_rotation(az, dep)
        v = pts - t
        rng = np.linalg.norm(v, axis=-1)
        c = v @ Rbc
        th = np.arctan2(np.hypot(c[..., 0], c[..., 1]), c[..., 2])
        uv = L.project(c)
        ok = (th < np.radians(85)) & (uv[..., 0] >= 0) & (uv[..., 0] <= W) & (uv[..., 1] >= 0) & (uv[..., 1] <= H)
        sin_dep = -v[..., 2] / rng
        cos_inc = sin_dep if surface == 'ground' else np.sqrt(1 - sin_dep ** 2)
        gsd = rng * L.pixel_angle(th) / np.maximum(cos_inc, 1e-6)
        n += ok & (gsd <= gsd_max)
    return n

CMAP = matplotlib.colors.ListedColormap(['#f0efec', '#c9dcf3', '#6fa6e6', '#2a78d6', '#16477f'])

def panel(rows, title, cbar):
    fig, axs = plt.subplots(len(rows), len(depressions_deg), figsize=(3.4 * len(depressions_deg), 3.5 * len(rows)), squeeze=False)
    for i, (label, maps) in enumerate(rows):
        for j, dep in enumerate(depressions_deg):
            ax = axs[i, j]
            ax.pcolormesh(X, Y, maps[dep], cmap=CMAP, vmin=-0.5, vmax=4.5, shading='auto')
            ax.set_aspect('equal'); ax.plot(0, 0, 'k^', ms=4)
            for r in (10, 20, 30):
                ax.add_patch(plt.Circle((0, 0), r * height_m / 20, fill=False, color='#888', lw=0.5, ls=':'))
            if i == 0: ax.set_title(f'axis {dep:g} deg down', loc='left')
            if j == 0: ax.set_ylabel(f'{label}\nforward (m)')
            if i == len(rows) - 1: ax.set_xlabel('right (m)')
    sm = plt.cm.ScalarMappable(cmap=CMAP, norm=matplotlib.colors.Normalize(-0.5, 4.5))
    fig.colorbar(sm, ax=axs, ticks=range(5), fraction=0.012).set_label(cbar)
    fig.suptitle(title, x=0.01, ha='left')
    show(fig)

def share(n, k, r):
    return ((n >= k) & (R <= r)).sum() / (R <= r).sum()

## Cameras seeing each ground point

Dotted circles at 10, 20 and 30 m for a 20 m height, scaled with `height_m`.

In [ ]:
count = {dep: seen(dep, height_m, 'ground') for dep in depressions_deg}
panel([('ground', count)], f'Cameras seeing each ground point from {height_m:g} m', 'cameras seeing the point')
radii = [height_m * f for f in (0.5, 1.0, 1.5)]
tab = pd.DataFrame({dep: {**{f'>=2 cams, r<={r:g} m': share(n, 2, r) for r in radii},
                          f'>=3 cams, r<={radii[1]:g} m': share(n, 3, radii[1]),
                          'r where <2 cams starts (m)': R[n < 2].min()} for dep, n in count.items()}).T
tab.index.name = 'axis deg down'
ipy_display(tab.round(2))

## Cameras resolving each point at `gsd_m` or better

Ground, then the camera-facing side of a trunk at each of `trunk_heights_m`.

In [ ]:
rows = [('ground', {dep: seen(dep, height_m, 'ground', gsd_m) for dep in depressions_deg})]
for h in trunk_heights_m:
    rows.append((f'trunk {h:g} m up', {dep: seen(dep, height_m - h, 'trunk', gsd_m) for dep in depressions_deg}))
panel(rows, f'Cameras resolving each point at {gsd_m * 100:g} cm or better, from {height_m:g} m',
      f'cameras at GSD <= {gsd_m * 100:g} cm')
radii = [height_m * f for f in (0.5, 1.0, 1.5, 2.0)]
tab = pd.DataFrame({(label, dep): {f'>=1 / >=2, r<={r:g} m': f'{share(n, 1, r):.2f} / {share(n, 2, r):.2f}' for r in radii}
                    for label, maps in rows for dep, n in maps.items()}).T
tab.index.names = ['surface', 'axis deg down']
ipy_display(tab)